# 03 · The LSTM as a Tool for an LLM

The trained LSTM becomes a **function the LLM can call**. The LLM decides when to run it, reads the structured
result and explains it — optionally grounded in documents retrieved from a **persistent local Chroma** vector store.

1. Wrap the model as a tool and run a minimal tool-calling loop
2. Add RAG: ingest `docs/ecg_reference.md` into Chroma and expose search as a second tool
3. Use the same `Agent` the FastAPI app serves
4. Call the running FastAPI service over HTTP

**OpenAI (`gpt-4o-mini`) is the agentic core**: it decides which tool to call and writes the answer. Our LSTM is one
of its tools. If the OpenAI call fails, the agent switches to Gemini (`gemini-2.5-flash`) through Google's
OpenAI-compatible endpoint. Requires `OPENAI_API_KEY` (and optionally `GEMINI_API_KEY`) in `.env`
(see `.env.example`) and the model from `02_lstm_training.ipynb`. All recordings used here are DS2 test patients
that the model never saw during training.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "mitdb"
MODEL_DIR = ROOT / "models"

In [2]:
import json
import numpy as np

from openai import OpenAI

from app.config import get_settings
from app.ecg import mitbih
from app.ecg.classifier import ECGClassifier

settings = get_settings()
endpoint = settings.llm_endpoint()          # LLM_PROVIDER=openai -> OPENAI_API_KEY, gpt-4o-mini
llm = OpenAI(api_key=endpoint.api_key, base_url=endpoint.base_url)
clf = ECGClassifier(settings.resolve(settings.ecg_model_path), settings.resolve(settings.ecg_meta_path))
print(f"LLM: {endpoint.provider}:{endpoint.model} | ECG classes: {clf.classes}")

I0000 00:00:1794198107.010259 2388150 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


LLM: openai:gpt-4o-mini | ECG classes: ['N', 'S', 'V']


## 1. A minimal tool: `analyze_ecg`
The LLM cannot pass 20,000 samples as an argument, so recordings live in a registry and the tool takes an id.

In [3]:
mitbih.download(DATA_DIR)
recordings = {}
for rec_id, seconds in [("233", 60), ("100", 60)]:
    sig, _, _, fs = mitbih.load_record(DATA_DIR, rec_id)
    recordings[f"mitdb_{rec_id}"] = {"signal": sig[: seconds * fs], "fs": fs}

def analyze_ecg(ecg_id: str) -> dict:
    r = recordings.get(ecg_id)
    if r is None:
        return {"error": f"unknown ecg_id {ecg_id}", "available": list(recordings)}
    return clf.analyze(r["signal"], r["fs"])

# Direct call — this is exactly what the LLM will receive
print(json.dumps({k: v for k, v in analyze_ecg("mitdb_233").items() if k != "abnormal_beats"}, indent=2))

{
  "duration_s": 60.0,
  "n_beats_classified": 102,
  "heart_rate_bpm": 103.4,
  "rr_sdnn_ms": 116.0,
  "beat_counts": {
    "N": 73,
    "S": 0,
    "V": 29
  },
  "beat_percentages": {
    "N": 71.57,
    "S": 0.0,
    "V": 28.43
  },
  "class_descriptions": {
    "N": "Normal / bundle branch block / escape beat",
    "S": "Supraventricular ectopic beat",
    "V": "Ventricular ectopic beat"
  },
  "abnormal_beats_truncated": false,
  "signal_quality": {
    "ok": true,
    "low_quality_fraction": 0.0,
    "median_kurtosis": 4.63,
    "warning": null
  },
  "disclaimer": "Automated research model output; not a medical diagnosis."
}


In [4]:
ECG_TOOL = {
    "type": "function",
    "function": {
        "name": "analyze_ecg",
        "description": "Run the LSTM ECG beat classifier on a stored recording. Returns heart rate, RR variability, AAMI beat counts and abnormal beat times.",
        "parameters": {
            "type": "object",
            "properties": {"ecg_id": {"type": "string", "enum": list(recordings)}},
            "required": ["ecg_id"],
        },
    },
}

def run_with_tools(question: str, tools: list, handlers: dict, system: str, max_steps: int = 5) -> str:
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = llm.chat.completions.create(model=endpoint.model, messages=messages, tools=tools).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"→ LLM called {tc.function.name}({args})")
            result = handlers[tc.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return "max steps reached"

answer = run_with_tools(
    "Compare recordings mitdb_233 and mitdb_100. Which one shows more ectopic activity and what kind?",
    tools=[ECG_TOOL],
    handlers={"analyze_ecg": analyze_ecg},
    system="You are a cardiology research assistant. Use the analyze_ecg tool; never guess. Results are not a diagnosis.",
)
print(answer)

→ LLM called analyze_ecg({'ecg_id': 'mitdb_233'})


→ LLM called analyze_ecg({'ecg_id': 'mitdb_100'})


Here's a comparison of the recordings for **mitdb_233** and **mitdb_100** regarding ectopic activity:

### Recording: mitdb_233
- **Total Duration**: 60 seconds
- **Heart Rate**: 103.4 bpm
- **RR Variability (SDNN)**: 116.0 ms
- **Total Beats Classified**: 102
- **Normal Beats (N)**: 73 (71.57%)
- **Ventricular Ectopic Beats (V)**: 29 (28.43%)
- **Supraventricular Ectopic Beats (S)**: 0
- **Abnormal Beats**: 
  - Total of 29 ventricular ectopic beats detected at various times with high confidence. 

### Recording: mitdb_100
- **Total Duration**: 60 seconds
- **Heart Rate**: 73.9 bpm
- **RR Variability (SDNN)**: 37.6 ms
- **Total Beats Classified**: 73
- **Normal Beats (N)**: 73 (100%)
- **Ventricular Ectopic Beats (V)**: 0
- **Supraventricular Ectopic Beats (S)**: 0
- **Abnormal Beats**: None detected.

### Conclusion
**mitdb_233** shows significantly more ectopic activity, specifically **29 ventricular ectopic beats**, while **mitdb_100** contains **no ectopic beats**. Thus, mitdb_233

## 2. Add RAG over the persistent Chroma store
Embeddings default to a local MiniLM model (no API key). Data persists in `chroma_db/`.

In [5]:
from app.embeddings import Embedder
from app.ingest import chunk_text
from app.vector_store import VectorStore

store = VectorStore(settings, Embedder(settings))
text = (ROOT / "docs" / "ecg_reference.md").read_text()
chunks = chunk_text(text, settings.chunk_size, settings.chunk_overlap)
store.delete_source("ecg_reference.md")
print("Ingested chunks:", store.add_chunks("ecg_reference.md", chunks), "| collection size:", store.count())

for hit in store.search("What does a wide QRS complex indicate?", top_k=3):
    print(f"{hit['score']:.3f}  {hit['text'][:120]!r}")

Ingested chunks: 5 | collection size: 5


0.502  '- **N (normal and bundle branch block beats):** normal sinus beats, left bundle branch block (LBBB),\n  right bundle bran'
0.460  '## Normal ECG intervals (adult)\n\n- Heart rate: 60–100 bpm at rest. Below 60 bpm is bradycardia; above 100 bpm is tachyca'
0.362  'followed by a compensatory pause.\n- **F (fusion beats):** a fusion of a ventricular and a normal beat; morphology is int'


In [6]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Semantic search over cardiology reference documents.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
    },
}

answer = run_with_tools(
    "Analyze mitdb_233. Is the amount of ventricular ectopy clinically significant according to the reference documents?",
    tools=[ECG_TOOL, SEARCH_TOOL],
    handlers={"analyze_ecg": analyze_ecg, "search_knowledge_base": lambda query: {"results": store.search(query, top_k=4)}},
    system="You are a cardiology research assistant. Use analyze_ecg for recordings and search_knowledge_base for clinical thresholds; cite sources. Not a diagnosis.",
)
print(answer)

→ LLM called analyze_ecg({'ecg_id': 'mitdb_233'})
→ LLM called search_knowledge_base({'query': 'ventricular ectopy clinical significance thresholds'})


The analysis of the ECG recording identified **29 ventricular ectopic beats (V)** out of **102 total beats**, resulting in **28.43%** of the beats being classified as ventricular ectopy. Here are some relevant findings from the data:

1. **Heart Rate**: 103.4 bpm (tachycardic).
2. **RR Interval Variability (SDNN)**: 116 ms.
3. **Normal (N) Beats**: 73 beats (71.57%).
4. **Ventricular (V) Beats**: 29 beats (28.43%).

### Clinical Significance of Ventricular Ectopy

According to reference documents, **frequent ectopy**—which is commonly defined as more than **10% of beats**—warrants clinical evaluation, particularly if accompanied by symptoms such as syncope or chest pain. Specifically, frequent **premature ventricular contractions (PVCs)** or runs of three or more consecutive ventricular beats (considered non-sustained ventricular tachycardia) are significant concerns. The percentage of ventricular ectopy in this case (28.43%) exceeds this commonly used threshold (10%).

Thus, based on 

## 3. The production `Agent` (same code the FastAPI `/chat` endpoint runs)

In [7]:
from app.agent import Agent, ECGRegistry

registry = ECGRegistry()
sig, _, _, fs = mitbih.load_record(DATA_DIR, "221")
rec = registry.add(sig[: 60 * fs].tolist(), fs, name="mitdb_221 first minute")

agent = Agent(settings, store, registry, clf)
out = agent.chat("Summarise this ECG and explain any abnormal beats in plain language.", ecg_id=rec.id)
print("LLM:", out["llm"])
print("Tools used:", out["tool_calls"])
print("Sources:", out["sources"])
print(out["answer"])

LLM: {'provider': 'openai', 'model': 'gpt-4o-mini', 'fallback_used': False}
Tools used: [{'tool': 'analyze_ecg', 'arguments': {'ecg_id': '124ed00f3aad'}}]
Sources: []
The ECG recording analyzed is 60 seconds long and shows a heart rate of approximately 75.2 beats per minute. A total of 74 beats were classified during this period, with the following breakdown:

- **Normal beats (N)**: 63 beats (85.14%)
- **Supraventricular ectopic beats (S)**: 2 beats (2.7%)
- **Ventricular ectopic beats (V)**: 9 beats (12.16%)

### Explanation of Abnormal Beats:
1. **Ventricular Ectopic Beats (V)**: 
   - There were 9 ventricular ectopic beats detected, which means that some of the heartbeats originated from the ventricles instead of the normal pacemaker (the sinoatrial node). This can happen when the ventricles fire prematurely, leading to an irregular heartbeat. The model is highly confident in these classifications, with confidence levels ranging from 0.993 to 0.999 for these beats.
   - Specific in

## 4. Through the FastAPI service

Start the server in a terminal from the project root:

```bash
source env/bin/activate
uvicorn app.main:app --reload
```

In [8]:
import httpx

API = f"http://localhost:{settings.app_port}"
try:
    with httpx.Client(base_url=API, timeout=120) as http:
        print(http.get("/health").json())
        sig, _, _, fs = mitbih.load_record(DATA_DIR, "214")
        ecg_id = http.post("/ecg/upload", json={"signal": sig[: 30 * fs].tolist(), "fs": fs, "name": "mitdb_214"}).json()["ecg_id"]
        print("Direct analysis:", http.post(f"/ecg/{ecg_id}/analyze").json()["beat_counts"])
        r = http.post("/chat", json={"message": "What does this recording show? Is it concerning?", "ecg_id": ecg_id}).json()
        print(r["llm"], r["tool_calls"]); print(r["answer"])
except httpx.ConnectError:
    print(f"API not running at {API} — start it with `uvicorn app.main:app` and re-run this cell.")

{'status': 'ok', 'vector_collection': 'ecg_knowledge', 'ecg_model_loaded': True, 'llm': {'provider': 'openai', 'model': 'gpt-4o-mini', 'fallback': 'gemini:gemini-2.5-flash'}}


Direct analysis: {'N': 34, 'S': 0, 'V': 2}


{'provider': 'openai', 'model': 'gpt-4o-mini', 'fallback_used': False} [{'tool': 'analyze_ecg', 'arguments': {'ecg_id': '6815dcc2f9cf'}}]
The ECG recording shows a total of 36 beats classified over a duration of 30 seconds, with a heart rate of approximately 74.7 beats per minute. The analysis indicates the following beat counts:

- **Normal beats (N)**: 34 (94.44%)
- **Ventricular ectopic beats (V)**: 2 (5.56%)
- **Supraventricular ectopic beats (S)**: 0 (0.0%)

The two ventricular ectopic beats detected were classified with high confidence (96.3% and 99.6%). 

In simple terms:
- **Normal beats (N)** are typical heartbeats.
- **Ventricular ectopic beats (V)** are extra beats that originate from the ventricles and can sometimes indicate an underlying issue, but they can also occur in healthy individuals without causing concern.

The recording quality is good, with no significant noise or artifacts detected.

While the presence of ventricular ectopic beats can be concerning depending on